In [1]:
# Import and project paths
from pathlib import Path
import sqlite3
import pandas as pd

# project root = parent of notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent

DB_PATH = PROJECT_ROOT / "db" / "app.db"
SCHEMA_PATH = PROJECT_ROOT / "db" / "schema.sql"
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DB_PATH exists:", DB_PATH.exists(), DB_PATH)
print("SCHEMA_PATH exists:", SCHEMA_PATH.exists(), SCHEMA_PATH)
print("DATA_RAW_DIR exists:", DATA_RAW_DIR.exists(), DATA_RAW_DIR)

PROJECT_ROOT: /Users/jonathanma/Desktop/DS Projects/diffusion-topic-evaluation
DB_PATH exists: True /Users/jonathanma/Desktop/DS Projects/diffusion-topic-evaluation/db/app.db
SCHEMA_PATH exists: True /Users/jonathanma/Desktop/DS Projects/diffusion-topic-evaluation/db/schema.sql
DATA_RAW_DIR exists: True /Users/jonathanma/Desktop/DS Projects/diffusion-topic-evaluation/data/raw


In [2]:
# inspect DB and schema
conn = sqlite3.connect(DB_PATH)

tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name
""", conn)

print("Existing tables:")
display(tables)

conn.close()

Existing tables:


,name
0,document_embeddings
1,documents


In [4]:
conn = sqlite3.connect(DB_PATH)

for table_name in ["documents", "document_embeddings"]:
    print(f"\nSchema for table: {table_name}")
    schema_df = pd.read_sql_query(f"PRAGMA table_info({table_name});", conn)
    display(schema_df)

conn.close()


Schema for table: documents


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,NaN,1
1,1,source,TEXT,0,NaN,0
2,2,source_doc_id,TEXT,0,NaN,0
3,3,title,TEXT,0,NaN,0
4,4,abstract,TEXT,0,NaN,0
5,5,publication_year,INTEGER,0,NaN,0
6,6,publication_date,TEXT,0,NaN,0
7,7,journal,TEXT,0,NaN,0
8,8,clean_text,TEXT,0,NaN,0
9,9,created_at,TIMESTAMP,0,CURRENT_TIMESTAMP,0



Schema for table: document_embeddings


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,NaN,1
1,1,document_id,INTEGER,0,NaN,0
2,2,embedding,BLOB,0,NaN,0
3,3,model_name,TEXT,0,NaN,0
4,4,created_at,TIMESTAMP,0,CURRENT_TIMESTAMP,0


In [6]:
# inspect raw files
raw_files = [f for f in sorted(DATA_RAW_DIR.glob("*")) if f.is_file()]

print(f"Number of files in data/raw: {len(raw_files)}")
for f in raw_files:
    print(f.name)

Number of files in data/raw: 1
.gitkeep


In [43]:
import requests
import xml.etree.ElementTree as ET
import math

# PubMed / NCBI E-utilities base
EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# Change these
NCBI_TOOL = "diffusion_topic_evolution"
NCBI_EMAIL = "REMOVED_EMAIL"   
NCBI_API_KEY = None  

# Small pilot query for debugging
PUBMED_QUERY = "covid-19[Title/Abstract] AND 2020[pdat]"

# search settings
SEARCH_RETMAX = 200
FETCH_BATCH_SIZE = 25
SLEEP_SECONDS = 0.8

print("Query:", PUBMED_QUERY)
print("SEARCH_RETMAX:", SEARCH_RETMAX)
print("FETCH_BATCH_SIZE:", FETCH_BATCH_SIZE)
print("SLEEP_SECONDS:", SLEEP_SECONDS)

Query: covid-19[Title/Abstract] AND 2020[pdat]
SEARCH_RETMAX: 200
FETCH_BATCH_SIZE: 25
SLEEP_SECONDS: 0.8


In [40]:
search_params = {
    "db": "pubmed",
    "term": PUBMED_QUERY,
    "retmax": SEARCH_RETMAX,
    "retmode": "json",
    "tool": NCBI_TOOL,
    "email": NCBI_EMAIL,
}

if NCBI_API_KEY:
    search_params["api_key"] = NCBI_API_KEY

search_url = f"{EUTILS_BASE}/esearch.fcgi"
search_resp = requests.get(search_url, params=search_params, timeout=30)
search_resp.raise_for_status()

search_data = search_resp.json()
id_list = search_data["esearchresult"].get("idlist", [])
total_count = int(search_data["esearchresult"].get("count", "0"))

print("Total matching records on PubMed:", total_count)
print("PMIDs retrieved this pass:", len(id_list))
print("First 10 PMIDs:", id_list[:10])

Total matching records on PubMed: 80733
PMIDs retrieved this pass: 200
First 10 PMIDs: ['39931522', '39911268', '39022319', '39022317', '38046875', '38046874', '38046872', '38046869', '38046867', '37928102']


In [20]:
# fetch pubmed records as xml
fetch_params = {
    "db": "pubmed",
    "id": ",".join(id_list),
    "retmode": "xml",
    "tool": NCBI_TOOL,
    "email": NCBI_EMAIL,
}

if NCBI_API_KEY:
    fetch_params["api_key"] = NCBI_API_KEY

fetch_url = f"{EUTILS_BASE}/efetch.fcgi"
fetch_resp = requests.get(fetch_url, params=fetch_params, timeout=30)
fetch_resp.raise_for_status()

xml_text = fetch_resp.text

print(xml_text[:1000])

<?xml version="1.0" ?>
<!DOCTYPE PubmedArticleSet PUBLIC "-//NLM//DTD PubMedArticle, 1st January 2025//EN" "https://dtd.nlm.nih.gov/ncbi/pubmed/out/pubmed_250101.dtd">
<PubmedArticleSet>
<PubmedArticle><MedlineCitation Status="PubMed-not-MEDLINE" Owner="NLM"><PMID Version="1">39931522</PMID><DateRevised><Year>2025</Year><Month>02</Month><Day>13</Day></DateRevised><Article PubModel="Electronic-eCollection"><Journal><ISSN IssnType="Print">2398-502X</ISSN><JournalIssue CitedMedium="Print"><Volume>5</Volume><PubDate><Year>2020</Year></PubDate></JournalIssue><Title>Wellcome open research</Title><ISOAbbreviation>Wellcome Open Res</ISOAbbreviation></Journal><ArticleTitle>Study protocol: Comparison of different risk prediction modelling approaches for COVID-19 related death using the OpenSAFELY platform.</ArticleTitle><Pagination><StartPage>243</StartPage><MedlinePgn>243</MedlinePgn></Pagination><ELocationID EIdType="pii" ValidYN="Y">243</ELocationID><ELocationID EIdType="doi" ValidYN="Y">10.1

In [21]:
root = ET.fromstring(xml_text)

articles = root.findall(".//PubmedArticle")
print("Parsed PubmedArticle count:", len(articles))

Parsed PubmedArticle count: 10


In [22]:
# extract text
def safe_text(elem):
    return elem.text.strip() if elem is not None and elem.text is not None else None


def extract_abstract_text(article):
    abstract_nodes = article.findall(".//Abstract/AbstractText")
    if not abstract_nodes:
        return None

    parts = []
    for node in abstract_nodes:
        label = node.attrib.get("Label")
        text = "".join(node.itertext()).strip()
        if text:
            parts.append(f"{label}: {text}" if label else text)

    return " ".join(parts) if parts else None


def extract_pub_date(article):
    pub_date = article.find(".//JournalIssue/PubDate")
    if pub_date is None:
        return None, None

    year = safe_text(pub_date.find("Year"))
    month = safe_text(pub_date.find("Month"))
    day = safe_text(pub_date.find("Day"))

    publication_date = "-".join([x for x in [year, month, day] if x])
    publication_year = int(year) if year and year.isdigit() else None

    return publication_date or None, publication_year

In [23]:
records = []

for article in articles:
    pmid = safe_text(article.find(".//PMID"))
    title = "".join(article.find(".//ArticleTitle").itertext()).strip() if article.find(".//ArticleTitle") is not None else None
    abstract = extract_abstract_text(article)
    journal = safe_text(article.find(".//Journal/Title"))
    publication_date, publication_year = extract_pub_date(article)

    records.append({
        "source": "pubmed",
        "source_doc_id": pmid,
        "title": title,
        "abstract": abstract,
        "publication_year": publication_year,
        "publication_date": publication_date,
        "journal": journal,
    })

print(f"Parsed records: {len(records)}")
records[:2]

Parsed records: 10


[{'source': 'pubmed',
  'source_doc_id': '39931522',
  'title': 'Study protocol: Comparison of different risk prediction modelling approaches for COVID-19 related death using the OpenSAFELY platform.',
  'abstract': 'On March 11th 2020, the World Health Organization characterised COVID-19 as a pandemic. Responses to containing the spread of the virus have relied heavily on policies involving restricting contact between people. Evolving policies regarding shielding and individual choices about restricting social contact will rely heavily on perceived risk of poor outcomes from COVID-19. In order to make informed decisions, both individual and collective, good predictive models are required. For outcomes related to an infectious disease, the performance of any risk prediction model will depend heavily on the underlying prevalence of infection in the population of interest. Incorporating measures of how this changes over time may result in important improvements in prediction model perfor

In [24]:
# making DF
df_raw = pd.DataFrame(records)

print("Shape:", df_raw.shape)
display(df_raw.head())

print("\nMissing values:")
display(df_raw.isna().sum())

print("\nColumns:")
print(df_raw.columns.tolist())

Shape: (10, 7)


,source,source_doc_id,title,abstract,publication_year,publication_date,journal
0,pubmed,39931522,Study protocol: Comparison of different risk p...,"On March 11th 2020, the World Health Organizat...",2020,2020,Wellcome open research
1,pubmed,39911268,"Mental health, sleep quality and quality of li...",BACKGROUND: Since the World Health Organizatio...,2020,2020,F1000Research
2,pubmed,39022319,Parent perspectives on food allergy management...,BACKGROUND: U.S. national emergency was declar...,2020,2020-Dec,Journal of food allergy
3,pubmed,39022317,When worlds collide: Food allergy and the COVI...,NaN,2020,2020-Dec,Journal of food allergy
4,pubmed,38046875,Should critically ill patients with COVID-19 b...,NaN,2020,2020-Dec,Critical care and resuscitation : journal of t...



Missing values:


source              0
source_doc_id       0
title               0
abstract            5
publication_year    0
publication_date    0
journal             0
dtype: int64


Columns:
['source', 'source_doc_id', 'title', 'abstract', 'publication_year', 'publication_date', 'journal']


In [25]:
# extraction and cleaning
df = df_raw.copy()

# drop rows with missing abstract or title
df = df.dropna(subset=["abstract", "title"])

# strip whitespace
for col in ["title", "abstract"]:
    df[col] = df[col].str.strip()

# remove duplicates (based on PubMed ID)
df = df.drop_duplicates(subset=["source_doc_id"])

# create clean_text (this is what we will embed later)
df["clean_text"] = df["title"] + " " + df["abstract"]

print("After cleaning:")
print("Shape:", df.shape)

display(df.head())

print("\nMissing values after cleaning:")
display(df.isna().sum())

After cleaning:
Shape: (5, 8)


,source,source_doc_id,title,abstract,publication_year,publication_date,journal,clean_text
0,pubmed,39931522,Study protocol: Comparison of different risk p...,"On March 11th 2020, the World Health Organizat...",2020,2020,Wellcome open research,Study protocol: Comparison of different risk p...
1,pubmed,39911268,"Mental health, sleep quality and quality of li...",BACKGROUND: Since the World Health Organizatio...,2020,2020,F1000Research,"Mental health, sleep quality and quality of li..."
2,pubmed,39022319,Parent perspectives on food allergy management...,BACKGROUND: U.S. national emergency was declar...,2020,2020-Dec,Journal of food allergy,Parent perspectives on food allergy management...
5,pubmed,38046874,COVID-19 critical illness in Sweden: character...,Objective: During the coronavirus disease 2019...,2020,2020-Dec,Critical care and resuscitation : journal of t...,COVID-19 critical illness in Sweden: character...
8,pubmed,38046867,Geolocated Twitter-based population mobility i...,"Using geotagged Twitter data in Victoria, we c...",2020,2020-Dec,Critical care and resuscitation : journal of t...,Geolocated Twitter-based population mobility i...



Missing values after cleaning:


source              0
source_doc_id       0
title               0
abstract            0
publication_year    0
publication_date    0
journal             0
clean_text          0
dtype: int64

In [28]:
conn = sqlite3.connect(DB_PATH)

cols = [
    "source",
    "source_doc_id",
    "title",
    "abstract",
    "publication_year",
    "publication_date",
    "journal",
    "clean_text",
]

df_to_insert = df[cols].copy()

# insert with duplicate protection
insert_sql = """
INSERT OR IGNORE INTO documents (
    source,
    source_doc_id,
    title,
    abstract,
    publication_year,
    publication_date,
    journal,
    clean_text
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
"""

conn.executemany(insert_sql, df_to_insert.values.tolist())
conn.commit()

# verify
count_df = pd.read_sql_query("SELECT COUNT(*) as n FROM documents", conn)
display(count_df)

conn.close()

,n
0,5


In [29]:
# inspection
conn = sqlite3.connect(DB_PATH)

preview_df = pd.read_sql_query("""
    SELECT
        id,
        source,
        source_doc_id,
        title,
        publication_year,
        publication_date,
        journal
    FROM documents
    ORDER BY id
""", conn)

display(preview_df)

conn.close()

,id,source,source_doc_id,title,publication_year,publication_date,journal
0,1,pubmed,39931522,Study protocol: Comparison of different risk p...,2020,2020,Wellcome open research
1,2,pubmed,39911268,"Mental health, sleep quality and quality of li...",2020,2020,F1000Research
2,3,pubmed,39022319,Parent perspectives on food allergy management...,2020,2020-Dec,Journal of food allergy
3,4,pubmed,38046874,COVID-19 critical illness in Sweden: character...,2020,2020-Dec,Critical care and resuscitation : journal of t...
4,5,pubmed,38046867,Geolocated Twitter-based population mobility i...,2020,2020-Dec,Critical care and resuscitation : journal of t...


In [31]:
conn = sqlite3.connect(DB_PATH)

year_counts = pd.read_sql_query("""
    SELECT publication_year, COUNT(*) AS n_docs
    FROM documents
    GROUP BY publication_year
    ORDER BY publication_year
""", conn)

display(year_counts)

conn.close()

,publication_year,n_docs
0,2020,5


In [32]:
conn = sqlite3.connect(DB_PATH)

dup_df = pd.read_sql_query("""
    SELECT source, source_doc_id, COUNT(*) AS n
    FROM documents
    GROUP BY source, source_doc_id
    HAVING COUNT(*) > 1
    ORDER BY n DESC, source_doc_id
""", conn)

display(dup_df)

conn.close()

,source,source_doc_id,n


In [66]:
# batch search 20
def safe_text(elem):
    return elem.text.strip() if elem is not None and elem.text is not None else None


def extract_abstract_text(article):
    abstract_nodes = article.findall(".//Abstract/AbstractText")
    if not abstract_nodes:
        return None

    parts = []
    for node in abstract_nodes:
        label = node.attrib.get("Label")
        text = "".join(node.itertext()).strip()
        if text:
            parts.append(f"{label}: {text}" if label else text)

    return " ".join(parts) if parts else None


def extract_journal_pub_date(article):
    pub_date = article.find(".//JournalIssue/PubDate")
    if pub_date is None:
        return None, None

    year = safe_text(pub_date.find("Year"))
    month = safe_text(pub_date.find("Month"))
    day = safe_text(pub_date.find("Day"))

    publication_date = "-".join([x for x in [year, month, day] if x])
    publication_year = int(year) if year and year.isdigit() else None

    return publication_date or None, publication_year


def extract_article_date(article):
    article_date = article.find(".//Article/ArticleDate")
    if article_date is None:
        return None, None

    year = safe_text(article_date.find("Year"))
    month = safe_text(article_date.find("Month"))
    day = safe_text(article_date.find("Day"))

    article_date_str = "-".join([x for x in [year, month, day] if x])
    article_year = int(year) if year and year.isdigit() else None

    return article_date_str or None, article_year


def parse_pubmed_xml_to_records(xml_text):
    root = ET.fromstring(xml_text)
    articles = root.findall(".//PubmedArticle")

    records = []
    for article in articles:
        pmid = safe_text(article.find(".//PMID"))
        title_node = article.find(".//ArticleTitle")
        title = "".join(title_node.itertext()).strip() if title_node is not None else None
        abstract = extract_abstract_text(article)
        journal = safe_text(article.find(".//Journal/Title"))

        journal_pub_date, journal_pub_year = extract_journal_pub_date(article)
        article_date, article_year = extract_article_date(article)

        # canonical project time field:
        # prefer journal publication year/date, then fall back to article date
        publication_year = journal_pub_year if journal_pub_year is not None else article_year
        publication_date = journal_pub_date if journal_pub_date is not None else article_date

        records.append({
            "source": "pubmed",
            "source_doc_id": pmid,
            "title": title,
            "abstract": abstract,
            "publication_year": publication_year,
            "publication_date": publication_date,
            "journal": journal,
            "article_date": article_date,
            "article_year": article_year,
            "journal_pub_date": journal_pub_date,
            "journal_pub_year": journal_pub_year,
        })

    return records

In [58]:
#21
from requests.exceptions import ChunkedEncodingError, RequestException

def fetch_pubmed_batch(pmids, max_retries=4, base_sleep=1.0):
    fetch_params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml",
        "tool": NCBI_TOOL,
        "email": NCBI_EMAIL,
    }

    if NCBI_API_KEY:
        fetch_params["api_key"] = NCBI_API_KEY

    fetch_url = f"{EUTILS_BASE}/efetch.fcgi"

    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(fetch_url, params=fetch_params, timeout=60)
            resp.raise_for_status()
            return resp.text
        except (ChunkedEncodingError, RequestException) as e:
            print(f"  attempt {attempt}/{max_retries} failed: {type(e).__name__} - {e}")
            if attempt == max_retries:
                raise
            time.sleep(base_sleep * attempt)

    raise RuntimeError("fetch_pubmed_batch failed unexpectedly")

# actual fetching

In [67]:
# 22
import time
all_records = []

n_batches = math.ceil(len(id_list) / FETCH_BATCH_SIZE)
print("Number of fetch batches:", n_batches)

for i in range(0, len(id_list), FETCH_BATCH_SIZE):
    batch_num = i // FETCH_BATCH_SIZE + 1
    pmid_batch = id_list[i:i + FETCH_BATCH_SIZE]

    print(f"Fetching batch {batch_num}/{n_batches} with {len(pmid_batch)} PMIDs...")
    xml_text = fetch_pubmed_batch(pmid_batch)
    batch_records = parse_pubmed_xml_to_records(xml_text)
    all_records.extend(batch_records)

    time.sleep(SLEEP_SECONDS)

print("Total parsed records:", len(all_records))

Number of fetch batches: 8
Fetching batch 1/8 with 25 PMIDs...
Fetching batch 2/8 with 25 PMIDs...
Fetching batch 3/8 with 25 PMIDs...
Fetching batch 4/8 with 25 PMIDs...
Fetching batch 5/8 with 25 PMIDs...
Fetching batch 6/8 with 25 PMIDs...
Fetching batch 7/8 with 25 PMIDs...
Fetching batch 8/8 with 25 PMIDs...
Total parsed records: 195


In [68]:
#23
df_raw = pd.DataFrame(all_records)

print("Raw batch shape:", df_raw.shape)
display(df_raw.head())

print("\nMissing values:")
display(df_raw.isna().sum())

print("\nPublication year counts:")
display(df_raw["publication_year"].value_counts(dropna=False).sort_index())

Raw batch shape: (195, 11)


,source,source_doc_id,title,abstract,publication_year,publication_date,journal,article_date,article_year,journal_pub_date,journal_pub_year
0,pubmed,39931522,Study protocol: Comparison of different risk p...,"On March 11th 2020, the World Health Organizat...",2020,2020,Wellcome open research,2024-12-16,2024.0,2020,2020
1,pubmed,39911268,"Mental health, sleep quality and quality of li...",BACKGROUND: Since the World Health Organizatio...,2020,2020,F1000Research,2024-07-29,2024.0,2020,2020
2,pubmed,39022319,Parent perspectives on food allergy management...,BACKGROUND: U.S. national emergency was declar...,2020,2020-Dec,Journal of food allergy,2020-12-01,2020.0,2020-Dec,2020
3,pubmed,39022317,When worlds collide: Food allergy and the COVI...,NaN,2020,2020-Dec,Journal of food allergy,2020-12-01,2020.0,2020-Dec,2020
4,pubmed,38046875,Should critically ill patients with COVID-19 b...,NaN,2020,2020-Dec,Critical care and resuscitation : journal of t...,2023-10-18,2023.0,2020-Dec,2020



Missing values:


source               0
source_doc_id        0
title                0
abstract            33
publication_year     0
publication_date     0
journal              0
article_date        29
article_year        29
journal_pub_date     0
journal_pub_year     0
dtype: int64


Publication year counts:


publication_year
2020    96
2021    58
2022    34
2023     7
Name: count, dtype: int64

In [69]:
df = df_raw.copy()

# keep only rows with usable text fields
df = df.dropna(subset=["title", "abstract", "publication_year"])

# standardize text
for col in ["title", "abstract", "journal", "publication_date"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# remove obviously empty strings that became "nan"
df = df[
    (df["title"].str.len() > 0) &
    (df["abstract"].str.len() > 0) &
    (df["title"].str.lower() != "nan") &
    (df["abstract"].str.lower() != "nan")
].copy()

# de-duplicate within this batch
df = df.drop_duplicates(subset=["source", "source_doc_id"])

# create clean text for later embedding
df["clean_text"] = df["title"] + " " + df["abstract"]

print("Cleaned batch shape:", df.shape)
print("\nMissing values after cleaning:")
display(df.isna().sum())

display(df.head())

Cleaned batch shape: (162, 12)

Missing values after cleaning:


source               0
source_doc_id        0
title                0
abstract             0
publication_year     0
publication_date     0
journal              0
article_date        29
article_year        29
journal_pub_date     0
journal_pub_year     0
clean_text           0
dtype: int64

,source,source_doc_id,title,abstract,publication_year,publication_date,journal,article_date,article_year,journal_pub_date,journal_pub_year,clean_text
0,pubmed,39931522,Study protocol: Comparison of different risk p...,"On March 11th 2020, the World Health Organizat...",2020,2020,Wellcome open research,2024-12-16,2024.0,2020,2020,Study protocol: Comparison of different risk p...
1,pubmed,39911268,"Mental health, sleep quality and quality of li...",BACKGROUND: Since the World Health Organizatio...,2020,2020,F1000Research,2024-07-29,2024.0,2020,2020,"Mental health, sleep quality and quality of li..."
2,pubmed,39022319,Parent perspectives on food allergy management...,BACKGROUND: U.S. national emergency was declar...,2020,2020-Dec,Journal of food allergy,2020-12-01,2020.0,2020-Dec,2020,Parent perspectives on food allergy management...
5,pubmed,38046874,COVID-19 critical illness in Sweden: character...,Objective: During the coronavirus disease 2019...,2020,2020-Dec,Critical care and resuscitation : journal of t...,2023-10-18,2023.0,2020-Dec,2020,COVID-19 critical illness in Sweden: character...
8,pubmed,38046867,Geolocated Twitter-based population mobility i...,"Using geotagged Twitter data in Victoria, we c...",2020,2020-Dec,Critical care and resuscitation : journal of t...,2023-10-18,2023.0,2020-Dec,2020,Geolocated Twitter-based population mobility i...


In [71]:
conn = sqlite3.connect(DB_PATH)

cols = [
    "source",
    "source_doc_id",
    "title",
    "abstract",
    "publication_year",
    "publication_date",
    "journal",
    "clean_text",
    "article_date",
    "article_year",
    "journal_pub_date",
    "journal_pub_year",
]

df_to_insert = df[cols].copy()

insert_sql = """
INSERT OR IGNORE INTO documents (
    source,
    source_doc_id,
    title,
    abstract,
    publication_year,
    publication_date,
    journal,
    clean_text,
    article_date,
    article_year,
    journal_pub_date,
    journal_pub_year
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

conn.executemany(insert_sql, df_to_insert.values.tolist())
conn.commit()

count_df = pd.read_sql_query("SELECT COUNT(*) as n FROM documents", conn)
display(count_df)

conn.close()

,n
0,162


In [63]:
conn = sqlite3.connect(DB_PATH)

year_counts = pd.read_sql_query("""
    SELECT publication_year, COUNT(*) AS n_docs
    FROM documents
    GROUP BY publication_year
    ORDER BY publication_year
""", conn)

dup_df = pd.read_sql_query("""
    SELECT source, source_doc_id, COUNT(*) AS n
    FROM documents
    GROUP BY source, source_doc_id
    HAVING COUNT(*) > 1
""", conn)

print("Counts by year:")
display(year_counts)

print("\nDuplicate source/source_doc_id rows:")
display(dup_df)

conn.close()

Counts by year:


,publication_year,n_docs
0,2020,90
1,2021,47
2,2022,25



Duplicate source/source_doc_id rows:


,source,source_doc_id,n


In [70]:
df_raw = pd.DataFrame(all_records)

print("Raw shape:", df_raw.shape)
display(df_raw[[
    "source_doc_id",
    "title",
    "journal_pub_year",
    "article_year",
    "publication_year",
    "journal_pub_date",
    "article_date"
]].head(10))

print("\nCanonical publication_year counts:")
display(df_raw["publication_year"].value_counts(dropna=False).sort_index())

Raw shape: (195, 11)


,source_doc_id,title,journal_pub_year,article_year,publication_year,journal_pub_date,article_date
0,39931522,Study protocol: Comparison of different risk p...,2020,2024.0,2020,2020,2024-12-16
1,39911268,"Mental health, sleep quality and quality of li...",2020,2024.0,2020,2020,2024-07-29
2,39022319,Parent perspectives on food allergy management...,2020,2020.0,2020,2020-Dec,2020-12-01
3,39022317,When worlds collide: Food allergy and the COVI...,2020,2020.0,2020,2020-Dec,2020-12-01
4,38046875,Should critically ill patients with COVID-19 b...,2020,2023.0,2020,2020-Dec,2023-10-18
5,38046874,COVID-19 critical illness in Sweden: character...,2020,2023.0,2020,2020-Dec,2023-10-18
6,38046872,Aerosol generation during surgical tracheostom...,2020,2023.0,2020,2020-Dec,2023-10-18
7,38046869,Feasibility and safety of angiotensin II admin...,2020,2023.0,2020,2020-Dec,2023-10-18
8,38046867,Geolocated Twitter-based population mobility i...,2020,2023.0,2020,2020-Dec,2023-10-18
9,37928102,Effect of awake prone positioning in hypoxaemi...,2023,2020.0,2023,2023-Nov,2020-09-24



Canonical publication_year counts:


publication_year
2020    96
2021    58
2022    34
2023     7
Name: count, dtype: int64

In [72]:
conn = sqlite3.connect(DB_PATH)

check_df = pd.read_sql_query("""
SELECT
    source_doc_id,
    publication_year,
    journal_pub_year,
    article_year
FROM documents
LIMIT 10
""", conn)

display(check_df)

conn.close()

,source_doc_id,publication_year,journal_pub_year,article_year
0,39931522,2020,2020,2024.0
1,39911268,2020,2020,2024.0
2,39022319,2020,2020,2020.0
3,38046874,2020,2020,2023.0
4,38046867,2020,2020,2023.0
5,37671353,2020,2020,2023.0
6,37649990,2020,2020,2022.0
7,33117894,2020,2020,2023.0
8,36753237,2020,2020,NaN
9,36753227,2020,2020,NaN


# 1000 cell search

In [73]:
TARGET_PMIDS = 1000
SEARCH_PAGE_SIZE = 200
FETCH_BATCH_SIZE = 25
SLEEP_SECONDS = 0.8

print("TARGET_PMIDS:", TARGET_PMIDS)
print("SEARCH_PAGE_SIZE:", SEARCH_PAGE_SIZE)
print("FETCH_BATCH_SIZE:", FETCH_BATCH_SIZE)
print("SLEEP_SECONDS:", SLEEP_SECONDS)

TARGET_PMIDS: 1000
SEARCH_PAGE_SIZE: 200
FETCH_BATCH_SIZE: 25
SLEEP_SECONDS: 0.8


In [74]:
all_pmids = []
retstart = 0

while len(all_pmids) < TARGET_PMIDS:
    page_retmax = min(SEARCH_PAGE_SIZE, TARGET_PMIDS - len(all_pmids))

    search_params = {
        "db": "pubmed",
        "term": PUBMED_QUERY,
        "retstart": retstart,
        "retmax": page_retmax,
        "retmode": "json",
        "tool": NCBI_TOOL,
        "email": NCBI_EMAIL,
    }

    if NCBI_API_KEY:
        search_params["api_key"] = NCBI_API_KEY

    search_url = f"{EUTILS_BASE}/esearch.fcgi"
    search_resp = requests.get(search_url, params=search_params, timeout=30)
    search_resp.raise_for_status()

    search_data = search_resp.json()
    batch_pmids = search_data["esearchresult"].get("idlist", [])
    total_count = int(search_data["esearchresult"].get("count", "0"))

    if not batch_pmids:
        print("No more PMIDs returned; stopping early.")
        break

    all_pmids.extend(batch_pmids)
    all_pmids = list(dict.fromkeys(all_pmids))  # preserve order, drop duplicates

    print(
        f"retstart={retstart:4d} | fetched={len(batch_pmids):3d} | "
        f"collected={len(all_pmids):4d} / target={TARGET_PMIDS} | total_available={total_count}"
    )

    retstart += len(batch_pmids)
    time.sleep(SLEEP_SECONDS)

print("\nFinal PMID count collected:", len(all_pmids))
print("First 10 PMIDs:", all_pmids[:10])
print("Last 10 PMIDs:", all_pmids[-10:])

retstart=   0 | fetched=200 | collected= 200 / target=1000 | total_available=80733
retstart= 200 | fetched=200 | collected= 400 / target=1000 | total_available=80733
retstart= 400 | fetched=200 | collected= 600 / target=1000 | total_available=80733
retstart= 600 | fetched=200 | collected= 800 / target=1000 | total_available=80733
retstart= 800 | fetched=200 | collected=1000 / target=1000 | total_available=80733

Final PMID count collected: 1000
First 10 PMIDs: ['39931522', '39911268', '39022319', '39022317', '38046875', '38046874', '38046872', '38046869', '38046867', '37928102']
Last 10 PMIDs: ['34173618', '34173616', '34173615', '34173614', '34173611', '34173610', '34173609', '34173607', '34173606', '34173605']
